# Buildings Tweedie XGBoost model

Builds, evaluates and scores a Tweedie XGBoost (variance power 1.5, matching the hackathon scoring) for the buildings insurance line, mirroring the validated pipelines in Motor_Bridget and Contents_Bridget. Reads the training master from `workings/` (persisted 2026-10-20), trains a 4-fold GroupKFold model with exposure weights, evaluates out-of-fold predictions, scores the 2025 `test_buildings` file (weather matched on address_id + policy year), and assembles the combined v4 submission with motor + buildings + contents all model-scored.

Data source: https://www.kaggle.com/competitions/iaa-ai-assisted-hackathon-2026-part-1

In [0]:
# Import / install libraries

%pip install xgboost openpyxl

   
#### Load data

In [0]:
from pyspark.sql import functions as F
import os

# FAST START: the buildings TRAINING master was built and persisted in Motor_Bridget
# (workings/policies_buildings_master.parquet, 2026-10-20) — read it directly instead of
# rebuilding. Raw datasets load lazily for the 2025 test master and submission steps.
workings_dir = "/Workspace/Users/bridget.macdonnell1@gmail.com/hackathon-project/workings"
data_dir = "/Workspace/Users/bridget.macdonnell1@gmail.com/hackathon-project/data"

df_policies_buildings_master = spark.read.parquet(os.path.join(workings_dir, "policies_buildings_master.parquet"))
print(f"Buildings master: {df_policies_buildings_master.count():,} rows, {len(df_policies_buildings_master.columns)} columns (read from workings/)")

dataframes = {}
for file_name in sorted(f for f in os.listdir(data_dir) if f.endswith(".parquet")):
    name = file_name.replace(".parquet", "")
    dataframes[name] = spark.read.parquet(os.path.join(data_dir, file_name))
    globals()[f"df_{name}"] = dataframes[name]
print(f"Raw datasets loaded lazily: {', '.join(sorted(dataframes))}")

   
#### Buildings feature selection

In [0]:
# Feature selection for the buildings Tweedie XGBoost — derived from Bridget's contents
# list (2026-10-20): same inclusion/exclusion choices applied to the columns that exist
# for buildings, plus the buildings-specific risk features. AI DRAFT LIST — Bridget to
# review/override with a 1/0 list if desired.
# Excluded as in contents: written_premium, discount_applied, customer_service_contacts,
# renewal_decision, all claims-derived columns (leakage), demographics IDs (surname,
# date_of_birth, email_domain, marketing_consent, document_preference, previous_insurer,
# campaign_code, customer_since_year).
# NOTE: the seven weather features (blank in the contents list) are INCLUDED — weather
# is joined on address_id + policy year, so 2025 test rows get 2025 weather.
# Contents-only covers that do NOT exist for buildings: tenure, security_level,
# high_value_items, works_from_home, lodgers, bicycle_cover.

BUILDINGS_FEATURES = [
    "Year",                                      # derived from period_start
    # exposure (written_premium excluded, as in contents)
    "exposure",
    # policy & property cover (buildings-specific features included)
    "policyholder_age", "sum_insured", "accidental_damage_cover", "construction_type",
    "occupancy", "days_unoccupied", "smoke_alarms", "mortgage",
    "voluntary_excess", "compulsory_excess",
    # policy admin
    "channel", "payment_method", "missed_payments_12m", "new_business", "renewal_count",
    "days_quote_to_inception",
    # customer
    "occupation", "marital_status", "household_size", "credit_band", "prior_claims_5y",
    "licence_points", "years_at_address",
    # address & property
    "branch_id", "marketing_segment", "region", "urban_density", "flood_zone",
    "distance_to_river_km", "elevation_m", "soil_type", "year_built", "property_type",
    "roof_type", "basement", "trees_within_5m", "listed_building", "bedrooms",
    "energy_rating", "solar_panels",
    # weather (joined on address_id + policy year)
    "hail_days", "wind_exposure_index", "freezing_days", "dry_summer_index",
    "wildfire_index", "rainfall_mm", "mean_temperature_c",
]

TARGET = "ultimate_claims"
GROUP_COL = "customer_id"              # GroupKFold grouping: all years of one customer stay together
TRACK_COLS = ["policy_id", "row_id"]   # for attaching predictions; never model inputs
EVAL_COL = "claim_indicator"           # diagnostic only (decile claim rates), not a feature

# Every listed column must exist in the master (Year is derived here, not in the master)
_missing = [c for c in BUILDINGS_FEATURES + [TARGET, GROUP_COL, EVAL_COL, *TRACK_COLS]
            if c != "Year" and c not in df_policies_buildings_master.columns]
assert not _missing, f"Listed columns missing from the master: {_missing}"

buildings_model = (df_policies_buildings_master
               .withColumn("Year", F.year("period_start"))
               # EXPOSURE FIX (2026-10-21): the raw exposure column mixes scales (547 rows
               # carry day counts like 365 instead of fractions of a year). Recompute from
               # the period dates: days in force / days in calendar year (366 in leap-year
               # 2024). Days in force is 2-365 across the master, so this is a proper [0,1]
               # exposure — it feeds both the feature and the Tweedie sample weight.
               .withColumn("exposure",
                           F.least((F.datediff("period_end", "period_start") + F.lit(1)) /
                                   F.when(F.year("period_start") % 4 == 0, F.lit(366)).otherwise(F.lit(365)),
                                   F.lit(1.0)))
               # CASING FIX (2026-10-21): construction_type has 11 raw levels that are really
               # 5 (Brick/BRICK/brick, stone/STONE, t/T, w/W, non_standard/NON_STANDARD).
               # lower(trim()) merges them; rare case variants were routing validation rows
               # into near-zero-prediction leaves (root cause of the fold-2 deviance blow-up).
               .withColumn("construction_type", F.lower(F.trim(F.col("construction_type"))))
               .withColumn(TARGET, F.coalesce(F.col(TARGET), F.lit(0.0)))   # no claims -> 0
               .select(*TRACK_COLS, GROUP_COL, *BUILDINGS_FEATURES, TARGET, EVAL_COL))

# One pass over the assembled dataset: size, target behaviour, feature missingness
s = buildings_model.agg(
    F.count(F.lit(1)).alias("_rows"),
    F.sum(F.when(F.col(TARGET) > 0, 1).otherwise(0)).alias("_claim_rows"),
    F.round(F.sum(TARGET), 2).alias("_target_sum"),
    *[F.sum(F.col(c).isNull().cast("int")).alias(c) for c in BUILDINGS_FEATURES],
).first()

n_rows = s["_rows"]
print(f"Buildings modeling dataset: {n_rows:,} rows | {len(BUILDINGS_FEATURES)} features | target = {TARGET} (nulls->0)")
print(f"Rows with target > 0: {s['_claim_rows']:,} ({100.0*s['_claim_rows']/n_rows:.1f}%) | total target sum: {s['_target_sum']:,.0f}")
print(f"Zero-target rows: {n_rows - s['_claim_rows']:,} ({100.0*(n_rows - s['_claim_rows'])/n_rows:.1f}%)")

feat_nulls = {c: s[c] for c in BUILDINGS_FEATURES if s[c] and s[c] > 0}
print("\nFeatures with missing values (left as-is — XGBoost handles them natively):")
for c, k in sorted(feat_nulls.items(), key=lambda x: -x[1]):
    print(f"  {c:<22} {100.0*k/n_rows:6.2f}%")

from collections import Counter
_dtypes = {c: t for c, t in buildings_model.dtypes if c in BUILDINGS_FEATURES}
print(f"\nFeature dtypes: {dict(Counter(_dtypes.values()))}")

display(buildings_model.limit(10))

   
#### Tweedie XGBoost modeling (buildings)

4-fold GroupKFold (grouped by customer_id) Tweedie XGBoost on the buildings master, mirroring the motor and contents pipelines. The Tweedie variance power is fixed at 1.5 to match the hackathon scoring, and models are evaluated with mean Tweedie deviance at p=1.5 — the same loss form the competition uses.

In [0]:
import pandas as pd

# Convert the buildings modeling dataset to pandas for XGBoost training:
# strings -> category (used natively via enable_categorical=True), booleans -> int.
# The same conversions, with category vocabularies learned from TRAINING only at
# scoring time, must be reused when scoring the test file so features stay aligned.

pdf = buildings_model.toPandas()

CAT_FEATURES = [c for c in BUILDINGS_FEATURES if str(pdf[c].dtype) == "object"]
BOOL_FEATURES = [c for c in BUILDINGS_FEATURES if pdf[c].dtype == bool]
# Nullable booleans (e.g. smoke_alarms) arrive as object dtype after the parquet
# round-trip and would otherwise be treated as categories with boolean values, which
# XGBoost's categorical path rejects — convert them to numeric 0/1/NaN instead
# (XGBoost treats NaN as missing natively).
NULLABLE_BOOL_FEATURES = [c for c in CAT_FEATURES
                          if set(map(type, pdf[c].dropna().unique())) <= {bool}]
CAT_FEATURES = [c for c in CAT_FEATURES if c not in NULLABLE_BOOL_FEATURES]
for c in CAT_FEATURES:
    pdf[c] = pdf[c].astype("category")
for c in BOOL_FEATURES:
    pdf[c] = pdf[c].astype("int8")
for c in NULLABLE_BOOL_FEATURES:
    pdf[c] = pdf[c].astype("float")

X = pdf[BUILDINGS_FEATURES]
y = pdf[TARGET].to_numpy(dtype=float)
w = pdf["exposure"].to_numpy(dtype=float)   # exposure weight: Tweedie pure premium model
groups = pdf[GROUP_COL]

print(f"Training matrix: {X.shape[0]:,} rows x {X.shape[1]} features")
print(f"Categorical ({len(CAT_FEATURES)}): {', '.join(CAT_FEATURES)}")
print(f"Boolean -> int ({len(BOOL_FEATURES)}): {', '.join(BOOL_FEATURES)}")
print(f"Nullable boolean -> 0/1/NaN ({len(NULLABLE_BOOL_FEATURES)}): {', '.join(NULLABLE_BOOL_FEATURES)}")
print(f"Exposure weight: min {w.min():.4f} | max {w.max():.2f} | zero: {(w == 0).sum():,} | >1: {(w > 1).sum():,} (known data issue, EDA later)")
display(pdf.head(5))

In [0]:
import numpy as np
import xgboost as xgb
from sklearn.model_selection import GroupKFold

# GroupKFold (4 folds): all policy-years of the same customer stay in one fold,
# so no customer is ever in both train and validation (avoids leakage across years).
# The hackathon scores with Tweedie power 1.5, so tweedie_variance_power is FIXED
# at 1.5 and each fold is evaluated with mean Tweedie deviance at p=1.5 — the same
# loss form the competition uses (XGBoost's built-in metric is a log-likelihood,
# not deviance).

POWER = 1.5          # Tweedie variance power — matches hackathon scoring (HUMAN NOTED)
N_FOLDS = 4

# PREDICTION FLOOR (2026-10-21): Tweedie unit deviance at p=1.5 grows as ~2*y/sqrt(pred)
# for small predictions, so one moderate claim on a near-zero predicted premium produced
# unit deviances of 1e8 (fold 2 = 2,216, of which 95.6% came from 5 rows with preds ~1e-9).
# Flooring at 1.0 (~0.5% of the 186 training mean pure premium) caps that blow-up while
# leaving genuinely low-risk rows (decile-1 mean pred ~20) essentially untouched.
PRED_FLOOR = 1.0

XGB_PARAMS = dict(
    objective="reg:tweedie",
    tweedie_variance_power=POWER,
    tree_method="hist",
    enable_categorical=True,
    max_depth=6,
    learning_rate=0.05,
    n_estimators=500,
    subsample=0.8,
    colsample_bytree=0.8,
    min_child_weight=5,
    reg_lambda=1.0,
    random_state=42,
    n_jobs=-1,
)

def tweedie_deviance(y_true, y_pred, p=POWER):
    """Mean Tweedie unit deviance (1<p<2) — the loss form the hackathon scores on (p=1.5)."""
    y_true = np.maximum(np.asarray(y_true, dtype=float), 0.0)
    y_pred = np.maximum(np.asarray(y_pred, dtype=float), 1e-9)
    unit = 2.0 * (
        np.power(y_true, 2.0 - p) / ((1.0 - p) * (2.0 - p))
        - y_true * np.power(y_pred, 1.0 - p) / (1.0 - p)
        + np.power(y_pred, 2.0 - p) / (2.0 - p)
    )
    return float(np.mean(unit))

gkf = GroupKFold(n_splits=N_FOLDS)
oof = pd.Series(np.nan, index=pdf.index, name="pred")
fold_rows = []
gain_total = {}   # feature -> summed gain across the 4 fold models

for k, (tr, va) in enumerate(gkf.split(X, y, groups=groups), start=1):
    model = xgb.XGBRegressor(**XGB_PARAMS)
    model.fit(X.iloc[tr], y[tr], sample_weight=w[tr])
    pred = np.maximum(model.predict(X.iloc[va]), PRED_FLOOR)
    oof.iloc[va] = pred

    score = model.get_booster().get_score(importance_type="gain")
    for f, g in score.items():
        gain_total[f] = gain_total.get(f, 0.0) + g

    dev = tweedie_deviance(y[va], pred)
    fold_rows.append({"fold": k, "train_rows": len(tr), "val_rows": len(va),
                      "tweedie_deviance_p1.5": dev,
                      "mean_pred": float(pred.mean()), "mean_actual": float(y[va].mean())})
    print(f"Fold {k}: {len(tr):,} train | {len(va):,} val | deviance(p=1.5) {dev:.4f} | "
          f"mean pred {pred.mean():.3f} vs actual {y[va].mean():.3f}")

fold_summary = pd.DataFrame(fold_rows)
print(f"\nMean Tweedie deviance (p=1.5) across folds: {fold_summary['tweedie_deviance_p1.5'].mean():.4f}")
display(fold_summary)

In [0]:
import os

# Out-of-fold (OOF) evaluation: every row is scored by a fold model that never saw it.
# Decile check: rows sorted into deciles by predicted pure premium — actual claims and
# claim rate should rise across deciles. Feature importance (gain) feeds the
# EDA/refinement loop. All results exported to Excel for review.

excel_dir = "/Workspace/Users/bridget.macdonnell1@gmail.com/hackathon-project/Excel files"

pdf["oof_pred"] = oof
pdf["pred_decile"] = pd.qcut(pdf["oof_pred"], 10, labels=False, duplicates="drop") + 1

decile_tbl = (pdf.groupby("pred_decile")
              .agg(n=("oof_pred", "size"),
                   mean_pred=("oof_pred", "mean"),
                   mean_actual=(TARGET, "mean"),
                   claim_rate=(EVAL_COL, "mean"))
              .reset_index())

mono = (decile_tbl["mean_actual"].diff().dropna() >= 0).mean()
lift = decile_tbl["mean_actual"].iloc[-1] / pdf[TARGET].mean()
print(f"Decile monotonicity (share of steps non-decreasing in actual claims): {mono:.0%}")
print(f"Top-decile lift (mean actual claims, top decile vs overall): {lift:.2f}x")
display(decile_tbl)

imp = (pd.DataFrame({"feature": list(gain_total.keys()), "total_gain": list(gain_total.values())})
       .sort_values("total_gain", ascending=False).reset_index(drop=True))
imp["gain_share"] = imp["total_gain"] / imp["total_gain"].sum()
print("Top 15 features by gain:")
display(imp.head(15))

out_path = os.path.join(excel_dir, "buildings_tweedie_xgb_results.xlsx")
with pd.ExcelWriter(out_path) as xw:
    decile_tbl.to_excel(xw, sheet_name="decile_lift", index=False)
    imp.to_excel(xw, sheet_name="feature_importance", index=False)
    fold_summary.to_excel(xw, sheet_name="cv_folds", index=False)
print(f"Results exported to: {out_path}")

In [0]:
import numpy as np
import pandas as pd
from sklearn.model_selection import GroupKFold

# ---- Investigate the fold-2 deviance outlier (2,216 vs 105-665 for the other folds) ----
# Re-derives the deterministic GroupKFold split (NO retraining) and recomputes per-row
# unit deviance at p=1.5 from the OOF predictions already cached on pdf, so we can see
# exactly which validation rows produced fold 2's huge score.

POWER = 1.5

def unit_deviance(y_true, y_pred, p=POWER):
    """Per-row Tweedie unit deviance (1<p<2) — same form as the fold scoring."""
    y_true = np.maximum(np.asarray(y_true, dtype=float), 0.0)
    y_pred = np.maximum(np.asarray(y_pred, dtype=float), 1e-9)
    return 2.0 * (
        np.power(y_true, 2.0 - p) / ((1.0 - p) * (2.0 - p))
        - y_true * np.power(y_pred, 1.0 - p) / (1.0 - p)
        + np.power(y_pred, 2.0 - p) / (2.0 - p)
    )

# Recreate the exact fold assignment (GroupKFold is deterministic given row order)
gkf = GroupKFold(n_splits=N_FOLDS)
fold_of_row = pd.Series(0, index=pdf.index)
for k, (tr, va) in enumerate(gkf.split(X, y, groups=pdf[GROUP_COL]), start=1):
    fold_of_row.iloc[va] = k
pdf["fold"] = fold_of_row
pdf["row_deviance"] = unit_deviance(pdf[TARGET], pdf["oof_pred"])

# Sanity check: per-fold stats should reproduce fold_summary (confirms the re-derived
# folds line up with the cached OOF predictions)
chk = (pdf.groupby("fold")
         .agg(n=(TARGET, "size"), mean_actual=(TARGET, "mean"),
              mean_pred=("oof_pred", "mean"), deviance=("row_deviance", "mean")))
print("Per-fold recheck (should match fold_summary from the CV cell):")
display(chk)

# How concentrated is fold-2's deviance?
f2 = pdf[pdf["fold"] == 2].sort_values("row_deviance", ascending=False)
tot = f2["row_deviance"].sum()
print(f"\nFold 2: {len(f2):,} rows | mean deviance with all rows: {tot/len(f2):,.1f}")
dev_ex_top10 = (tot - f2["row_deviance"].head(10).sum()) / (len(f2) - 10)
print(f"Mean deviance if the top 10 rows were excluded: {dev_ex_top10:,.1f}")
for k in (1, 5, 10, 50):
    share = f2["row_deviance"].head(k).sum() / tot
    print(f"  top {k:>2} rows contribute {share:5.1%} of fold-2 deviance")

cols = ["fold", "policy_id", "row_id", "customer_id", TARGET, "oof_pred", "row_deviance",
        "exposure", "sum_insured", "construction_type", "occupancy", "flood_zone", "region", "Year"]

print("\nTop 10 fold-2 rows by unit deviance (the rows behind the 2,216):")
display(f2.head(10)[cols])

# Context: where did the biggest claims in the whole dataset land, and did the model
# predict them at all? Also check the other folds' tails for comparison.
print("15 largest claims overall — which fold absorbed each:")
display(pdf.nlargest(15, TARGET)[cols])

for k in (1, 3, 4):
    fk = pdf[pdf["fold"] == k]
    fk_top = fk.nlargest(10, "row_deviance")
    print(f"Fold {k}: top-10 deviance rows = {fk_top['row_deviance'].sum():,.0f} "
          f"({fk_top['row_deviance'].sum()/fk['row_deviance'].sum():.1%} of fold) | "
          f"largest claim {fk[TARGET].max():,.0f} | claims >10k: {(fk[TARGET] > 10_000).sum()}")
print(f"Fold 2: claims >10k: {(pdf.loc[pdf['fold'] == 2, TARGET] > 10_000).sum()}")

   
#### Test-file scoring & submission (buildings)

Refit the buildings Tweedie XGBoost on 100% of the training rows (the CV run was evaluation only), build the 2025 test buildings master with the same joins as training (weather matched on address_id + policy year, so 2025 test rows get 2025 weather), predict ultimate_claims, and fill the B-prefix rows of the sample_submission skeleton. Motor/contents rows (and any zero predictions) keep the dummy value 10 — the scorer does not accept zeros.

In [0]:
# Build the buildings TEST master for the 2025 policies, mirroring the training master
# joins: weather (address_id + policy year — 2025 test rows match 2025 weather),
# customers (customer_id), addresses (address_id). The test file has no claims data and
# no renewal_decision, so there is no claims join and NO renewed-only filter — every
# test row is scored (population mismatch vs the renewed-only training filter is a
# known flagged decision).

wx = (dataframes["weather"]
      .withColumnRenamed("address_id", "_wx_address_id")
      .withColumnRenamed("year", "_wx_year"))

test_buildings_master = (
    dataframes["test_buildings"]
    .join(wx,
          (F.col("address_id") == F.col("_wx_address_id")) &
          (F.year(F.col("period_start")) == F.col("_wx_year")),
          "left")
    .drop("_wx_address_id", "_wx_year")
    .join(dataframes["customers"], "customer_id", "left")
    .join(dataframes["addresses"], "address_id", "left")
)

test_buildings_model = (test_buildings_master
    .withColumn("Year", F.year("period_start"))
    # Same two fixes as the training master (cell 7): recompute exposure from the period
    # dates (2025 is not a leap year -> 365), and normalize construction_type casing.
    # Applied to BOTH train and test so the feature distributions stay aligned.
    .withColumn("exposure",
                F.least((F.datediff("period_end", "period_start") + F.lit(1)) /
                        F.when(F.year("period_start") % 4 == 0, F.lit(366)).otherwise(F.lit(365)),
                        F.lit(1.0)))
    .withColumn("construction_type", F.lower(F.trim(F.col("construction_type"))))
    .select("row_id", "policy_id", "customer_id", *BUILDINGS_FEATURES))

# Every training feature must be present in the test master
_t_missing = [c for c in BUILDINGS_FEATURES if c != "Year" and c not in test_buildings_model.columns]
assert not _t_missing, f"Features missing from the test master: {_t_missing}"

test_pdf = test_buildings_model.toPandas()
print(f"Test buildings master: {len(test_pdf):,} rows | {len(BUILDINGS_FEATURES)} features (same as training)")
print(f"Customer-feature nulls (join gap): {test_pdf['occupation'].isna().mean():.1%}")

# Align categoricals with the TRAINING vocabularies; unseen test levels -> NaN.
# Booleans -> int, exactly as in the training prep.
X_test = test_pdf[BUILDINGS_FEATURES].copy()
_new_levels = {}
for c in CAT_FEATURES:
    cats = pdf[c].cat.categories
    _new_levels[c] = int((~test_pdf[c].dropna().isin(cats)).sum())
    X_test[c] = pd.Categorical(test_pdf[c], categories=cats)
for c in BOOL_FEATURES:
    X_test[c] = X_test[c].astype("int8")
for c in NULLABLE_BOOL_FEATURES:
    X_test[c] = test_pdf[c].astype("float")   # same 0/1/NaN conversion as training

_new = {c: k for c, k in _new_levels.items() if k > 0}
print(f"Unseen categorical levels (mapped to NaN): {_new if _new else 'none'}")
display(X_test.head(5))

In [0]:
import os
import numpy as np

# FINAL model: refit the CV configuration on 100% of the training rows — the CV run
# was evaluation only; the final model uses all training data before scoring the 2025
# buildings test rows (mirrors the contents instruction: predict test_buildings and
# add the results to the sample_submission skeleton).
final_model = xgb.XGBRegressor(**XGB_PARAMS)
final_model.fit(X, y, sample_weight=w)
test_pred = np.maximum(final_model.predict(X_test), PRED_FLOOR)   # same floor as the CV run

# Submission on the sample_submission skeleton (row_id + prediction). Buildings rows
# (row_id prefix B) get the model prediction; motor/contents rows have no model in
# this file and the scorer rejects zeros, so they — and any rounded-zero buildings
# prediction — keep the dummy value 10.
DUMMY = 10
assert test_pdf["row_id"].is_unique, "test_buildings row_id is not unique"

sub = dataframes["sample_submission"].select("row_id").toPandas()
buildings_map = pd.DataFrame({"row_id": test_pdf["row_id"], "prediction": np.round(test_pred, 2)})
sub = sub.merge(buildings_map, on="row_id", how="left")
sub["prediction"] = sub["prediction"].fillna(DUMMY)
sub.loc[sub["prediction"] == 0, "prediction"] = DUMMY

# Sanity checks: full skeleton, no nulls or zeros, every B row scored by the model
assert len(sub) == 626_639, f"unexpected row count: {len(sub):,}"
assert sub["prediction"].isna().sum() == 0 and (sub["prediction"] <= 0).sum() == 0

pref = sub["row_id"].str[0]
b_mask = pref == "B"
assert b_mask.sum() == len(test_pdf), "B-row count does not match test_buildings rows"
print(f"Submission rows: {len(sub):,} | by line: {pref.value_counts().to_dict()}")
print(f"Buildings rows scored by the model: {b_mask.sum():,} | "
      f"mean pred {sub.loc[b_mask, 'prediction'].mean():.2f} (training mean claims/row 186.3) | "
      f"min {sub.loc[b_mask, 'prediction'].min():.2f} | max {sub.loc[b_mask, 'prediction'].max():.2f}")
print(f"Dummy rows (M/C lines, no model in this file): {(~b_mask).sum():,} | "
      f"buildings zero-preds replaced by dummy: {int(sub.loc[b_mask, 'prediction'].eq(DUMMY).sum())}")

sub_dir = "/Workspace/Users/bridget.macdonnell1@gmail.com/hackathon-project/submissions"
os.makedirs(sub_dir, exist_ok=True)
sub_path = os.path.join(sub_dir, "buildings_tweedie_xgb_submission_v002.csv")  # v002 = exposure + casing + pred-floor fixes (v001 kept on disk)
sub.to_csv(sub_path, index=False)
print(f"\nSubmission saved to: {sub_path}")
display(sub.head(10))

   
#### Combined submission v5 (motor + buildings + contents)

Assemble the full submission with ALL THREE lines scored by Bridget's own models: M rows from the motor Tweedie XGBoost (v1 file), B rows from the rebuilt buildings model (v002 file — exposure recomputed from period dates, construction_type casing normalized, predictions floored at 1.0), C rows from the contents model (v001 file). v4 is left untouched as the previous submission; this writes a new v5 file. Each line is verified against its source file before saving.

In [0]:
import os
import numpy as np
import pandas as pd

# Combined v5: M <- motor model (v1), B <- buildings model v002 (rebuilt with the
# exposure recompute + construction_type casing fix + prediction floor), C <- contents
# model (v001). v4 is kept untouched; this writes a new v5 file. Sources are read from
# disk so the assembly is deterministic and reproducible after any restart.
sub_dir = "/Workspace/Users/bridget.macdonnell1@gmail.com/hackathon-project/submissions"

v4_base = pd.read_csv(os.path.join(sub_dir, "motor_buildings_contents_submission_v004.csv"))      # M + C model rows, B = old buildings model
buildings_sub = pd.read_csv(os.path.join(sub_dir, "buildings_tweedie_xgb_submission_v002.csv"))   # B model rows (fixed)
motor_v1 = pd.read_csv(os.path.join(sub_dir, "motor_tweedie_xgb_submission_v001.csv"))           # M model rows
contents_v1 = pd.read_csv(os.path.join(sub_dir, "contents_tweedie_xgb_submission_v001.csv"))     # C model rows

v5 = v4_base.merge(buildings_sub, on="row_id", how="left", suffixes=("", "_buildings"))
b_mask = v5["row_id"].str.startswith("B")
assert v5.loc[~b_mask, "prediction_buildings"].notna().all()
# only B rows take the new buildings predictions; M/C rows keep their existing values
v5.loc[b_mask, "prediction"] = v5.loc[b_mask, "prediction_buildings"]
v5 = v5[["row_id", "prediction"]]

# ---- verification ----
assert len(v5) == 626_639, f"unexpected row count: {len(v5):,}"
assert v5["prediction"].isna().sum() == 0 and (v5["prediction"] <= 0).sum() == 0
assert v5["row_id"].is_unique

pref = v5["row_id"].str[0]
counts = pref.value_counts().to_dict()
print(f"v5 rows: {len(v5):,} | by line: {counts}")

m_check = motor_v1.set_index("row_id")["prediction"]
mm = v5[pref == "M"]
assert np.allclose(mm["prediction"].to_numpy(), mm["row_id"].map(m_check).to_numpy()), "M rows differ from motor v1"
print(f"M rows identical to motor v1: {len(mm):,} | mean {mm['prediction'].mean():.2f}")

c_check = contents_v1.set_index("row_id")["prediction"]
cc = v5[pref == "C"]
assert np.allclose(cc["prediction"].to_numpy(), cc["row_id"].map(c_check).to_numpy()), "C rows differ from contents v001"
print(f"C rows identical to contents v001: {len(cc):,} | mean {cc['prediction'].mean():.2f}")

bb = v5[pref == "B"]
b_map = buildings_sub.set_index("row_id")["prediction"]
assert np.allclose(bb["prediction"].to_numpy(), bb["row_id"].map(b_map).to_numpy()), "B rows differ from buildings v002"
print(f"B rows scored by the rebuilt buildings model (v002): {len(bb):,} | "
      f"min {bb['prediction'].min():.2f} | mean {bb['prediction'].mean():.2f} | max {bb['prediction'].max():.2f}")

v5_path = os.path.join(sub_dir, "motor_buildings_contents_submission_v005.csv")
v5.to_csv(v5_path, index=False)
print(f"\nCombined submission saved to: {v5_path}")
display(v5.head(10))